# Stop Guessing, Start Measuring
## Evaluating RAG with synthetic test data

This notebook mirrors the **current** RAGAS docs, as of 19.06.2026. 
It does four things:

1. Builds a deliberately *simple* RAG system over the public **Hugging Face docs** corpus.
2. **Generates a synthetic test set** from those docs with RAGAS's **knowledge-graph** pipeline.
3. Runs the RAG over the test set.
4. **Evaluates** it with the five RAG metrics via the current `ragas.metrics.collections` API:
   **Context Precision, Context Recall, Faithfulness, Answer Relevancy, Noise Sensitivity.**


## 0. Setup

In [45]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import ragas
from ragas.testset import TestsetGenerator
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, NoiseSensitivity,
)
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
print("ragas", ragas.__version__, "- imports OK")


ragas 0.4.3 - imports OK


## 1. A tiny RAG over the Hugging Face docs

Deliberately minimal — OpenAI embeddings for retrieval, OpenAI chat for generation. The talk is
about *evaluating* the RAG, not building a fancy one. Swap in your own pipeline here later.

In [47]:
from datasets import load_dataset
from langchain_core.documents import Document

# Hugging Face docs corpus
ds = load_dataset("m-ric/huggingface_doc", split="train")
subset = ds.select(range(50))  # keep it small for the demo
documents = [Document(page_content=r["text"], metadata={"source": r.get("source", "")})
             for r in subset]
print(len(documents), "documents")


50 documents


### Peek at the source documents (Hugging Face docs)

Browse the corpus in the dataset viewer: **https://huggingface.co/datasets/m-ric/huggingface_doc**
— a scrape of the official docs at https://huggingface.co/docs . Each row is one page (`text`)
plus its `source`.

In [48]:
print("columns:", subset.column_names)
for r in subset.select(range(2)):
    print("\nsource:", r.get("source"))
    print(r["text"][:300], "...")


columns: ['text', 'source']

source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-f ...

source: huggingface/evaluate/blob/main/docs/source/choosing_a_metric.mdx
 Choosing a metric for your task

**So you've trained your model and want to see how well it’s doing on a dataset of your choice. Where do you start?**

There is no “one size fits all” approach to choosing an evaluation metric, but some good guidelines to keep in mind are:

## Categories of metrics
 ...


In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from openai import OpenAI

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
GEN_MODEL = "gpt-4o-mini"

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = [c.page_content for c in splitter.split_documents(documents)]
print(len(chunks), "chunks")

def embed(texts):
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)

# Embed the corpus once (this is the only bulk embedding call).
chunk_emb = embed(chunks)
chunk_emb /= (np.linalg.norm(chunk_emb, axis=1, keepdims=True) + 1e-9)


462 chunks


In [50]:
def retrieve(query, k=4):
    q = embed([query])[0]
    q /= (np.linalg.norm(q) + 1e-9)
    sims = chunk_emb @ q
    return [chunks[i] for i in sims.argsort()[::-1][:k]]

def rag_answer(query, k=4):
    ctx = retrieve(query, k)
    context = "\n\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(ctx))
    msg = [
        {"role": "system", "content": "Answer ONLY from the provided documents. Be concise. "
                                       "If the answer is not in the documents, say so."},
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {query}"},
    ]
    out = client.chat.completions.create(model=GEN_MODEL, messages=msg)
    return out.choices[0].message.content, ctx

ans, ctx = rag_answer("What is a tokenizer, in a single sentence?")
print(ans)


A tokenizer is a tool that performs preprocessing on input text to convert it into a format that can be easily processed by machine learning models, typically by breaking the text into tokens.


## 2. Generate a synthetic test set with RAGAS (knowledge graph)

RAGAS chunks the docs into a **knowledge graph** (nodes = chunks, edges = shared entities/relationships), then traverses it to manufacture diverse questions —
including **multi-hop** ones that join two pages, the edge cases a human would never hand-write.

Each generated row gives us **`user_input`**, **`reference`** (gold answer), and **`reference_contexts`** — the labelled test item we otherwise couldn't build by hand.

In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms, default_transforms_for_prechunked

generator = TestsetGenerator.from_langchain(
    ChatOpenAI(model="gpt-4o-mini"),
    OpenAIEmbeddings(model="text-embedding-3-small"),
)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_chunks = splitter.split_documents(documents)

kg = KnowledgeGraph()
for d in doc_chunks:
    kg.nodes.append(
        Node(
            type=NodeType.CHUNK,
            properties={"page_content": d.page_content, "document_metadata": d.metadata},
        )
    )

# Build the KG: extract entities/themes/summaries -> relationships
transforms = default_transforms_for_prechunked(generator.llm, generator.embedding_model)
apply_transforms(kg, transforms)
generator.knowledge_graph = kg

# Traverse the KG to synthesize queries (single-hop + multi-hop).
testset = generator.generate(testset_size=8)

df = testset.to_pandas()
df[["user_input", "reference"]]


Applying SummaryExtractor:   0%|          | 0/462 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/462 [00:00<?, ?it/s]

Node 96d4c6da-6f3e-49a5-9ad4-aa5eafefc756 does not have a summary. Skipping filtering.
Node 48b8d9c9-d92d-4206-a951-bc545ce0058d does not have a summary. Skipping filtering.
Node 9795f4d9-8622-468e-9837-d43b04893d6b does not have a summary. Skipping filtering.
Node b223ea93-ff5e-47b4-a9df-27e3a58b1436 does not have a summary. Skipping filtering.
Node a85b8397-64e8-4ec9-8aec-f2cba39997c9 does not have a summary. Skipping filtering.
Node 2405bdf5-e37a-425a-98dd-4489df9cc35f does not have a summary. Skipping filtering.
Node abe617e1-d0ab-4e6f-8225-a0a6c6aed46d does not have a summary. Skipping filtering.
Node dd2d77d6-47c1-453e-803e-d1553a391241 does not have a summary. Skipping filtering.
Node 391680cb-890b-4b8b-9169-788a629f05c6 does not have a summary. Skipping filtering.
Node bae52e95-1df9-403a-bf56-8696c65947e6 does not have a summary. Skipping filtering.
Node 4d6fcfc1-7771-495e-942d-eb4ed2fafa5f does not have a summary. Skipping filtering.
Node b8b0159d-ab5c-485f-98c2-a1c6240abbb8 d

Applying EmbeddingExtractor:   0%|          | 0/462 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/462 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

,user_input,reference
0,What steps should I follow to create an endpoint on Azure once it becomes available as a cloud provider?,"To create an endpoint, you will first log in and be directed to the Endpoint creation page. You will need to enter the Hugging Face Repo..."
1,How is Named Entity Recognition evaluated in machine learning?,"Named Entity Recognition is often evaluated using task-specific metrics, specifically with the seqeval metric."
2,What perplexity used for in generative tasks?,Perplexity can be used for evaluating different kinds of unsupervised generative tasks.
3,What steps must be taken to edit files in a repository and share a model on the Hugging Face Hub?,"To edit files in a repository, you can easily view the commit history and the differences between versions. Before sharing a model to th..."
4,What are the key contributions of T5v1.1 and Switch Transformers in the field of machine learning?,"T5v1.1, developed by Google AI, is a model that focuses on text-to-text transfer learning, as detailed in the paper 'Exploring the Limit..."
5,How does TimeSformer contribute to video understanding and what advantages does it have over traditional 3D convolutional networks?,TimeSformer contributes to video understanding by utilizing a convolution-free approach that relies solely on self-attention mechanisms ...
6,What is the significance of the image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers...,The image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png is signifi...
7,How can the stabilityai/stable-diffusion-xl-base-1.0 model be optimized for generating images with different styles using LoRAs?,The stabilityai/stable-diffusion-xl-base-1.0 model can be optimized for generating images with different styles by combining it with sty...
8,"What is the significance of DeiT in the context of vision-text dual encoder models, and how does it relate to the training of data-effic...","DeiT, which stands for 'Data-efficient image transformers,' is significant in the context of vision-text dual encoder models as it serve..."


### Peek at what came out — including a multi-hop question

In [52]:
import pandas as pd
pd.set_option("display.max_colwidth", 140)

# 'synthesizer_name' shows which query type produced each row (single-hop vs multi-hop).
cols = [c for c in ["user_input", "reference", "synthesizer_name"] if c in df.columns]
display(df[cols])

# Show one full row (question + gold answer + reference contexts).
row0 = df.iloc[-1]
print("Question:", row0["user_input"])
print("\nReference answer:", row0["reference"])
print("\n# reference_contexts:", len(row0["reference_contexts"]))


,user_input,reference,synthesizer_name
0,What steps should I follow to create an endpoint on Azure once it becomes available as a cloud provider?,"To create an endpoint, you will first log in and be directed to the Endpoint creation page. You will need to enter the Hugging Face Repo...",single_hop_specific_query_synthesizer
1,How is Named Entity Recognition evaluated in machine learning?,"Named Entity Recognition is often evaluated using task-specific metrics, specifically with the seqeval metric.",single_hop_specific_query_synthesizer
2,What perplexity used for in generative tasks?,Perplexity can be used for evaluating different kinds of unsupervised generative tasks.,single_hop_specific_query_synthesizer
3,What steps must be taken to edit files in a repository and share a model on the Hugging Face Hub?,"To edit files in a repository, you can easily view the commit history and the differences between versions. Before sharing a model to th...",multi_hop_abstract_query_synthesizer
4,What are the key contributions of T5v1.1 and Switch Transformers in the field of machine learning?,"T5v1.1, developed by Google AI, is a model that focuses on text-to-text transfer learning, as detailed in the paper 'Exploring the Limit...",multi_hop_abstract_query_synthesizer
5,How does TimeSformer contribute to video understanding and what advantages does it have over traditional 3D convolutional networks?,TimeSformer contributes to video understanding by utilizing a convolution-free approach that relies solely on self-attention mechanisms ...,multi_hop_abstract_query_synthesizer
6,What is the significance of the image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers...,The image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/img2img-init.png is signifi...,multi_hop_specific_query_synthesizer
7,How can the stabilityai/stable-diffusion-xl-base-1.0 model be optimized for generating images with different styles using LoRAs?,The stabilityai/stable-diffusion-xl-base-1.0 model can be optimized for generating images with different styles by combining it with sty...,multi_hop_specific_query_synthesizer
8,"What is the significance of DeiT in the context of vision-text dual encoder models, and how does it relate to the training of data-effic...","DeiT, which stands for 'Data-efficient image transformers,' is significant in the context of vision-text dual encoder models as it serve...",multi_hop_specific_query_synthesizer


Question: What is the significance of DeiT in the context of vision-text dual encoder models, and how does it relate to the training of data-efficient image transformers?

Reference answer: DeiT, which stands for 'Data-efficient image transformers,' is significant in the context of vision-text dual encoder models as it serves as a pretrained vision autoencoding model that can be utilized alongside various text autoencoding models, such as RoBERTa or BERT. The VisionTextDualEncoderModel allows for the initialization of a dual encoder model that combines these vision and text encoders, projecting their output embeddings into a shared latent space. This integration is crucial for aligning vision-text embeddings and enables the model to perform zero-shot vision tasks, such as image classification or retrieval. The training of DeiT emphasizes data efficiency and the use of attention mechanisms, as detailed in the paper 'Training data-efficient image transformers & distillation through atten

### Inspect & save the knowledge graph

In [53]:
# The generator exposes the KG it built. Attribute name can vary slightly by version,
# so we guard it.
try:
    kg = generator.knowledge_graph
    print("nodes:", len(kg.nodes), "| relationships:", len(kg.relationships))
    sample_node = kg.nodes[0]
    print("example node properties:", list(sample_node.properties.keys()))
    kg.save("hf_docs_kg.json")   # persist -> render this for the 'show them the KG' slide
    print("saved -> hf_docs_kg.json")
except AttributeError as e:
    print("KG attribute differs in this ragas version:", e)
    print("See docs.ragas.io/en/stable/concepts/test_data_generation/rag/")


nodes: 462 | relationships: 660
example node properties: ['page_content', 'document_metadata', 'summary', 'summary_embedding', 'themes', 'entities']
saved -> hf_docs_kg.json


### Visualize the knowledge graph

Nodes are document chunks; edges connect chunks RAGAS found **related** — sharing named
entities/themes, or being semantically similar (cosine similarity of their summaries). Those edges
are exactly what let it build **multi-hop** questions that span two chunks.

In [61]:
# %pip install -q pyvis networkx
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame

net = Network(height="100vh", width="100%", bgcolor="#ffffff", font_color="#222",
              notebook=True, cdn_resources="in_line")
# Node label = the same number as the matplotlib legend; hover (title) shows the full chunk.
for n, d in G.nodes(data=True):
    net.add_node(n, label=str(d.get("idx", "")), title=d.get("snippet", ""),
                 color="#6c8cff", size=16)
# Edge label = relationship type, shown directly on the edge (hover too).
for a, b, d in G.edges(data=True):
    rt = d.get("rtype", "")
    net.add_edge(a, b, label=rt, title=rt, color="#94a3b8", font={"size": 10})
net.force_atlas_2based(gravity=-40, spring_length=130)
net.show("hf_docs_kg.html")
IFrame("hf_docs_kg.html", width="100%", height=620)

hf_docs_kg.html


### How to make a unique test set based on your requirements

Personas + a weighted query distribution let you steer the mix (e.g. 40% entity single-hop /
40% theme single-hop / 20% multi-hop).

In [55]:
from ragas.testset.persona import Persona
from ragas.testset.synthesizers.single_hop.specific import SingleHopSpecificQuerySynthesizer
from ragas.testset.synthesizers.multi_hop.specific import MultiHopSpecificQuerySynthesizer

persona = Persona(
    name="HF developer",
    role_description="A developer integrating a Hugging Face library, asking practical how-to "
                     "and API questions.",
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="entities"), 0.4),
    (SingleHopSpecificQuerySynthesizer(llm=generator.llm, property_name="themes"),   0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator.llm),                            0.2),
]

# The KG was already built in the generation cell above, so we re-run generate() (not
# generate_with_langchain_docs) to apply the persona + distribution without rebuilding it.
# generator.persona_list = [persona]
# testset = generator.generate(testset_size=8, query_distribution=query_distribution)
print("personas + distribution configured (re-run generation to apply)")


personas + distribution configured (re-run generation to apply)


-------------------------
## 3. Run the RAG over the generated test set

In [56]:
records = []
for _, r in df.iterrows():
    response, retrieved = rag_answer(r["user_input"], k=4)
    records.append({
        "user_input": r["user_input"],
        "reference": r["reference"],
        "response": response,
        "retrieved_contexts": retrieved,
    })

eval_df = pd.DataFrame(records)
eval_df[["user_input", "response"]]


,user_input,response
0,What steps should I follow to create an endpoint on Azure once it becomes available as a cloud provider?,The documents do not provide specific steps on how to create an endpoint on Azure once it becomes available.
1,How is Named Entity Recognition evaluated in machine learning?,The documents do not provide specific details on how Named Entity Recognition is evaluated in machine learning.
2,What perplexity used for in generative tasks?,Perplexity is used for evaluating different kinds of (unsupervised) generative tasks.
3,What steps must be taken to edit files in a repository and share a model on the Hugging Face Hub?,"To edit files in a repository and share a model on the Hugging Face Hub, follow these steps:\n\n1. **Edit Files**: Use the repository in..."
4,What are the key contributions of T5v1.1 and Switch Transformers in the field of machine learning?,The documents do not specify the key contributions of T5v1.1 and Switch Transformers in the field of machine learning.
5,How does TimeSformer contribute to video understanding and what advantages does it have over traditional 3D convolutional networks?,TimeSformer contributes to video understanding by introducing a convolution-free approach that utilizes self-attention over both space a...
6,What is the significance of the image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers...,The significance of the image located at that URL is not provided in the documents.
7,How can the stabilityai/stable-diffusion-xl-base-1.0 model be optimized for generating images with different styles using LoRAs?,The stabilityai/stable-diffusion-xl-base-1.0 model can be optimized for generating images with different styles using LoRAs by combining...
8,"What is the significance of DeiT in the context of vision-text dual encoder models, and how does it relate to the training of data-effic...",The significance of DeiT (Data-efficient Image Transformers) in the context of vision-text dual encoder models is that it serves as a pr...


## 4. Score with the five RAGAS metrics

A separate **judge** LLM scores each row. Note how each metric reads a different slice of the
trajectory — that's what lets you localize a failure to *retrieval* vs *generation*.

| Metric | Reads | Good = | Catches |
|---|---|---|---|
| **Context Precision** | question, contexts, reference | high | retriever returning / ranking junk |
| **Context Recall** | question, contexts, reference | high | retriever missing needed chunks |
| **Faithfulness** | question, response, contexts | high | hallucination / unsupported claims |
| **Answer Relevancy** *(docs: Response Relevancy)* | question, response | high | evasive / off-topic answers |
| **Noise Sensitivity** | question, response, reference, contexts | **low** | junk context throwing it off |

*Naming:* the class is **`AnswerRelevancy`**; the docs display it as **Response Relevancy**. **`ContextPrecision`** here is the **reference-based** variant — it reads `reference` (not `response`). The sibling `ContextPrecisionWithoutReference` uses `response` instead, for systems with no gold answer.

In [57]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall, NoiseSensitivity,
)

aclient = AsyncOpenAI()
judge     = llm_factory("gpt-4o-mini", client=aclient)                            # evaluator ("judge") LLM
judge_emb = RagasOpenAIEmbeddings(client=aclient, model="text-embedding-3-small")  # AnswerRelevancy needs embeddings

faithfulness      = Faithfulness(llm=judge)
answer_relevancy  = AnswerRelevancy(llm=judge, embeddings=judge_emb)
context_precision = ContextPrecision(llm=judge)
context_recall    = ContextRecall(llm=judge)
noise_sensitivity = NoiseSensitivity(llm=judge)


In [58]:
# Top-level await works inside Jupyter. Each metric is an LLM-as-judge call.
async def score_row(row):
    return {
        "context_precision": (await context_precision.ascore(
            user_input=row["user_input"], reference=row["reference"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "context_recall": (await context_recall.ascore(
            user_input=row["user_input"], reference=row["reference"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "faithfulness": (await faithfulness.ascore(
            user_input=row["user_input"], response=row["response"],
            retrieved_contexts=row["retrieved_contexts"])).value,
        "answer_relevancy": (await answer_relevancy.ascore(
            user_input=row["user_input"], response=row["response"])).value,
        "noise_sensitivity": (await noise_sensitivity.ascore(
            user_input=row["user_input"], response=row["response"],
            reference=row["reference"], retrieved_contexts=row["retrieved_contexts"])).value,
    }

scores = []
for r in eval_df.to_dict("records"):
    scores.append(await score_row(r))   # top-level await is supported in Jupyter

results = pd.concat([eval_df, pd.DataFrame(scores)], axis=1)
results.to_csv("eval_results.csv", index=False)
results[["user_input", "context_precision", "context_recall",
         "faithfulness", "answer_relevancy", "noise_sensitivity"]]


,user_input,context_precision,context_recall,faithfulness,answer_relevancy,noise_sensitivity
0,What steps should I follow to create an endpoint on Azure once it becomes available as a cloud provider?,1.000000,1.000000,0.333333,0.000000,0.000000
1,How is Named Entity Recognition evaluated in machine learning?,0.000000,0.000000,1.000000,0.000000,1.000000
2,What perplexity used for in generative tasks?,0.250000,1.000000,1.000000,0.726485,0.500000
3,What steps must be taken to edit files in a repository and share a model on the Hugging Face Hub?,1.000000,1.000000,0.727273,0.985577,0.000000
4,What are the key contributions of T5v1.1 and Switch Transformers in the field of machine learning?,1.000000,0.666667,1.000000,0.000000,1.000000
5,How does TimeSformer contribute to video understanding and what advantages does it have over traditional 3D convolutional networks?,1.000000,0.750000,1.000000,0.891074,0.250000
6,What is the significance of the image located at https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers...,0.583333,0.000000,1.000000,0.000000,0.500000
7,How can the stabilityai/stable-diffusion-xl-base-1.0 model be optimized for generating images with different styles using LoRAs?,1.000000,1.000000,0.750000,1.000000,0.375000
8,"What is the significance of DeiT in the context of vision-text dual encoder models, and how does it relate to the training of data-effic...",1.000000,1.000000,0.636364,0.872533,0.181818


In [59]:
metric_cols = ["context_precision", "context_recall", "faithfulness",
               "answer_relevancy", "noise_sensitivity"]
results[metric_cols].mean().round(3)

context_precision    0.759
context_recall       0.713
faithfulness         0.827
answer_relevancy     0.497
noise_sensitivity    0.423
dtype: float64